# 06 — S3 (stretch, report only): touch features

Four sub-step ball-contact features per 5 fps step (per-team min player–ball distance over the 5 frames behind the step, peak change of ball velocity, team of the nearest player at that peak; PREREG §5.1) appended to exp_1's global vector. One seed (20261108), `exp3_ctrl` recipe and `L_ctrl` targets, 28 nested fits. Compared with `exp3_ctrl` seed 0. Never changes the candidate.

In [1]:
from pathlib import Path
import os, sys, json, time
ROOT = Path.cwd()
if not (ROOT / 'exp3_common.py').exists():
    raise RuntimeError('Run with working directory exp_jeremy/exp_3')
sys.path.insert(0, str(ROOT))
sys.dont_write_bytecode = True
import numpy as np
import pandas as pd
pd.set_option('display.width', 220)
pd.set_option('display.max_columns', 40)
import exp3_common as C
import exp1_common as C1
protocol = C.verify_prereg()
print('PREREG verified', protocol['prereg_sha256'][:16], '| frozen', protocol['frozen_at_utc'])
import exp3_protocol as P, exp3_touch as S, exp3_report as R, exp2_report as R2
assert all(len(list(P.prediction_folder(a).glob('*.npz'))) == 28 for a in P.MODEL_ARMS), 'primary fits must finish first'

PREREG verified cff444fff0106e9c | frozen 2026-11-08T16:53:50+00:00


In [2]:
t0 = time.time()
features = pd.DataFrame(S.build_feature_cache())
features.to_csv(C.RESULTS / 's3_feature_stats.csv', index=False)
print(features.groupby('kind').mean(numeric_only=True).round(4).to_string(), f'| {time.time() - t0:.0f}s')

             round  mean_home_min_dist  mean_peak_dv  peak_team_defined
kind                                                                   
original       0.0              1.6221        0.0551             0.5919
target_aug     1.5              1.9943        0.0510             0.3958
target_like    0.0              1.9900        0.0506             0.3975 | 15s


In [3]:
log_path = C.RESULTS / 'fit_log.txt'
def log(message, **_):
    print(message, flush=True)
    with log_path.open('a') as handle:
        handle.write(message + '\n')
t0 = time.time()
predictions = S.ensure_predictions(log=log)
log(f'ARM DONE {S.ARM} {time.time() - t0:.0f}s, fingerprint {S.fingerprint()}')

FIT exp3_touch pair_J03WMX_J03WN1 46.0s


FIT exp3_touch pair_J03WMX_J03WOH 43.6s


FIT exp3_touch pair_J03WMX_J03WOY 90.6s


FIT exp3_touch pair_J03WMX_J03WPY 43.6s


FIT exp3_touch pair_J03WMX_J03WQQ 43.7s


FIT exp3_touch pair_J03WMX_J03WR9 43.3s


... (19 outputs left out)

FIT exp3_touch outer_J03WPY 51.6s


FIT exp3_touch outer_J03WQQ 51.5s


FIT exp3_touch outer_J03WR9 52.1s


ARM DONE exp3_touch 1328s, fingerprint 17928a0edbb4


In [4]:
rows, deltas = [], []
for label_set in ('ctrl', 'provider'):
    for view in C.VIEWS:
        result = P.evaluate_predictions(predictions, S.ARM, view, label_set)
        C1.write_json(C.RESULTS / f'variant_{S.ARM}__{view}__{label_set}.json', result)
        both = pd.DataFrame(result['rows'] + R.variant_rows('exp3_ctrl', view, label_set))
        d = R2.delta(both, view, S.ARM, 'exp3_ctrl')
        deltas.append({'label_set': label_set, **d})
        rows += result['rows']
        print(f"{label_set:8s} {view:12s} touch {pd.DataFrame(result['rows']).spell_f1.mean():.4f} | delta vs exp3_ctrl {d['mean_delta']:+.4f} [{d['ci95'][0]:+.4f}, {d['ci95'][1]:+.4f}] wins {d['wins']}")
C1.write_json(C.RESULTS / 's3_touch.json', {'arm': S.ARM, 'fingerprint': S.fingerprint(), 'deltas_vs_exp3_ctrl_seed0': deltas,
                                            'rows': rows, 'report_only': True})

ctrl     original     touch 0.7399 | delta vs exp3_ctrl +0.0116 [-0.0040, +0.0256] wins 6


ctrl     target_like  touch 0.7373 | delta vs exp3_ctrl +0.0120 [-0.0040, +0.0269] wins 6


provider original     touch 0.5961 | delta vs exp3_ctrl +0.0111 [-0.0031, +0.0273] wins 5


provider target_like  touch 0.5969 | delta vs exp3_ctrl +0.0132 [-0.0010, +0.0296] wins 5
